In [3]:
#importing necessary libraries
import numpy as np 
import pandas as pd
import os
import re
import glob
from pathlib import Path

In [4]:
import sys
print(sys.version)


3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [5]:
DATA_DIR = r"C:\Users\Samreet\Desktop\DataAnalysis_Revision\DA_Course\Python\PythonHackathon_Sep2026\Python_Hackathon_Sep_2026\cardiac_failure_Dataset"
csv_files = glob.glob(DATA_DIR + "/*.csv") #Retrieve the directory full path
#csv_files=os.listdir(DATA_DIR)            #Retrieve the file name from the directory
csv_files                                  #show the list of all csv files along with their full path

['C:\\Users\\Samreet\\Desktop\\DataAnalysis_Revision\\DA_Course\\Python\\PythonHackathon_Sep2026\\Python_Hackathon_Sep_2026\\cardiac_failure_Dataset\\cardiac_complications.csv',
 'C:\\Users\\Samreet\\Desktop\\DataAnalysis_Revision\\DA_Course\\Python\\PythonHackathon_Sep2026\\Python_Hackathon_Sep_2026\\cardiac_failure_Dataset\\demography.csv',
 'C:\\Users\\Samreet\\Desktop\\DataAnalysis_Revision\\DA_Course\\Python\\PythonHackathon_Sep2026\\Python_Hackathon_Sep_2026\\cardiac_failure_Dataset\\hospitalization_discharge.csv',
 'C:\\Users\\Samreet\\Desktop\\DataAnalysis_Revision\\DA_Course\\Python\\PythonHackathon_Sep2026\\Python_Hackathon_Sep_2026\\cardiac_failure_Dataset\\labs.csv',
 'C:\\Users\\Samreet\\Desktop\\DataAnalysis_Revision\\DA_Course\\Python\\PythonHackathon_Sep2026\\Python_Hackathon_Sep_2026\\cardiac_failure_Dataset\\patienthistory.csv',
 'C:\\Users\\Samreet\\Desktop\\DataAnalysis_Revision\\DA_Course\\Python\\PythonHackathon_Sep2026\\Python_Hackathon_Sep_2026\\cardiac_failure_

In [6]:
print("Number of CSV files found:", len(csv_files))

Number of CSV files found: 7


In [7]:
# Count rows in each CSV in the dataset
for file in csv_files:
    df = pd.read_csv(file)
    print(os.path.basename(file), "->", len(df), "rows")

cardiac_complications.csv -> 2008 rows
demography.csv -> 2009 rows
hospitalization_discharge.csv -> 2008 rows
labs.csv -> 2008 rows
patienthistory.csv -> 2008 rows
patient_precriptions.csv -> 15362 rows
responsivenes.csv -> 2008 rows


In [8]:
## Seperate the columns and quick preview
csv_check=pd.read_csv(csv_files[5], sep=";").head(50)
csv_check

# To understand the structure of dataset before Data cleaning, we used two functions
#read_csv to read the csv data into a python dataframe
#head() will show the top few records for preview

,"inpatient_number,drug_name"
0,"857781,sulfotanshinone sodium injection"
1,"857781,Furosemide tablet"
2,"857781,Enoxaparin Sodium injection"
3,"857781,Meglumine Adenosine Cyclophosphate for ..."
4,"857781,Furosemide injection"
5,"857781,Milrinone injection"
6,"857781,Metoprolol Succinate Sustained-release ..."
7,"857781,Deslanoside injection"
8,"857781,Torasemide tablet"
9,"857781,Benazepril hydrochloride tablet"


In [9]:
#To read the csv file from the dataset(DEMOGRAPHY DATA FILE)
data = pd.read_csv(r"C:\Users\Samreet\Desktop\DataAnalysis_Revision\DA_Course\Python\PythonHackathon_Sep2026\Python_Hackathon_Sep_2026\cardiac_failure_Dataset\patient_precriptions.csv")

In [10]:
#To read the csv file and saved it copy as df_clean (DEMOGRAPHY DATA FILE)
#df_clean = pd.read_csv(r"C:\Users\Samreet\Desktop\DataAnalysis_Revision\DA_Course\Python\PythonHackathon_Sep2026\Python_Hackathon_Sep_2026\cardiac_failure_Dataset\patient_precriptions.csv")
presc_clean=data.copy()

In [11]:
# Preview the content of the Data file
print(presc_clean.head())
print(presc_clean.shape)
print(presc_clean.columns)
print(presc_clean.tail())
print(len(presc_clean))


   inpatient_number                                         drug_name
0            857781                  sulfotanshinone sodium injection
1            857781                                 Furosemide tablet
2            857781                       Enoxaparin Sodium injection
3            857781  Meglumine Adenosine Cyclophosphate for injection
4            857781                              Furosemide injection
(15362, 2)
Index(['inpatient_number', 'drug_name'], dtype='object')
       inpatient_number                     drug_name
15357            791864  Valsartan Dispersible tablet
15358            791864                Digoxin tablet
15359            791864         Deslanoside injection
15360            791864           Milrinone injection
15361            791864          Furosemide injection
15362


In [12]:
# To Strip spaces & lowercase all drug names in order to be consistent 
presc_clean['drug_name'] = presc_clean['drug_name'].str.strip().str.lower()


In [13]:
print(presc_clean.head(10))

   inpatient_number                                         drug_name
0            857781                  sulfotanshinone sodium injection
1            857781                                 furosemide tablet
2            857781                       enoxaparin sodium injection
3            857781  meglumine adenosine cyclophosphate for injection
4            857781                              furosemide injection
5            857781                               milrinone injection
6            857781     metoprolol succinate sustained-release tablet
7            857781                             deslanoside injection
8            857781                                 torasemide tablet
9            857781                   benazepril hydrochloride tablet


In [14]:
# Checking data type of patients id's (column is inpatient_number)
column_dtype = presc_clean['inpatient_number'].dtype
print(f"The column data type is: {column_dtype}")

# Identify non-numeric ids 
non_numeric_mask = pd.to_numeric(presc_clean['inpatient_number'], errors='coerce').isna()
non_numeric_ids = presc_clean[non_numeric_mask]
if non_numeric_ids.empty:
    print("All patient IDs are completely numeric.")
else:
    print(f"Found {len(non_numeric_ids)} row(s) with non-numeric or missing patient IDs.\n")
    print(non_numeric_ids)

The column data type is: int64
All patient IDs are completely numeric.


In [15]:
#To Verify to find any duplicate records and print the number of duplicated rows if any
duplicate_count = presc_clean.duplicated().sum()
if duplicate_count > 0:
    print(f"Yes, the dataset contains {duplicate_count} duplicate rows.")
else:
    print("No duplicate rows found in the dataset.")


No duplicate rows found in the dataset.


In [16]:
# Another set of verification to find all duplicate records based on both patient ID and drug name in data file
duplicate_patients = presc_clean[presc_clean.duplicated(subset=['inpatient_number', 'drug_name'], keep=False)]

if not duplicate_patients.empty:
    # Count how many duplicate rows exist per patient ID
    duplicate_counts_per_patient = duplicate_patients.groupby('inpatient_number').size() // 2
    
    print(f"Found {duplicate_counts_per_patient.count()} unique patient ID(s) with duplicate prescriptions.\n")
    print("--- Patient IDs and their number of duplicate entries ---")
    print(duplicate_counts_per_patient.to_string())
else:
    print("No patient IDs have duplicate entries for the same drug.")

No patient IDs have duplicate entries for the same drug.


In [17]:
# Rename the column permanently
presc_clean = presc_clean.rename(columns={'inpatient_number': 'patient_id'})

In [18]:
print(presc_clean[['patient_id', 'drug_name']].head())

   patient_id                                         drug_name
0      857781                  sulfotanshinone sodium injection
1      857781                                 furosemide tablet
2      857781                       enoxaparin sodium injection
3      857781  meglumine adenosine cyclophosphate for injection
4      857781                              furosemide injection


In [19]:
presc_clean.to_csv('patient_prescriptions_cleaned.csv', index=False)
print("\nCleaned dataset exported successfully as 'patient_prescriptions_cleaned.csv'!")


Cleaned dataset exported successfully as 'patient_prescriptions_cleaned.csv'!
